# Greffer un classifieur fastai

Ce notebook utilise explicitement `modelAAE_DROPOUT.py` depuis la branche `Arda` de `https://github.com/LucaLaFisca/Human-Centered-xAI`. Dans ce modèle, `forward` calcule `self.zi` puis retourne `self.linear(self.zi)`: la greffe remplace donc la tête `linear` par le classifieur fourni.

In [ ]:
import torch
from torch import nn

from tell_me_why import (
    HumanCenteredAAEConfig,
    build_human_centered_aae,
    graft_classifier_to_human_aae,
    ensure_human_centered_xai_repo,
    make_grafted_learner,
    resolve_human_centered_aae_path,
)

In [ ]:
# Clone la branche Arda si ../Human-Centered-xAI n'existe pas encore.
# ensure_human_centered_xai_repo()

resolve_human_centered_aae_path()

Exemple de construction du modèle. Cette cellule demande l'environnement fastai/PyTorch complet de `Human-Centered-xAI`.

```python
config = HumanCenteredAAEConfig(input_size=256, input_channels=3, encoding_dims=128, classes=2)
aae = build_human_centered_aae(config)

classifier = nn.Sequential(
    nn.Linear(config.encoding_dims, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, config.classes),
)

model = graft_classifier_to_human_aae(aae, classifier, freeze_aae_body=True)
```

Avec des `DataLoaders` fastai existants, le helper crée directement le `Learner`:

```python
learn = make_grafted_learner(
    dls,
    classifier,
    config=config,
    pretrained_weights="CL_AAE_model_128",
)
```